# Notebook 01 — Dataset Exploration
## AI-Based Multilingual Cyberbullying Detection

This notebook provides a full exploratory analysis of `train.csv` — the primary dataset used
to train the multimodal fusion model (m-BERT/MuRIL + ResNet50 + Context MLP).

**Contents:**
1. Dataset Overview
2. Schema Validation
3. Language Distribution
4. Label Distribution (Aggression, Repetition, Intent)
5. Text Length Analysis
6. Missing Value Analysis
7. Sample Records
8. Key Findings & Observations

## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import sys

# Make project root importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Paths
DATA_CSV  = os.path.join(PROJECT_ROOT, 'data', 'train.csv')
IMAGE_DIR = os.path.join(PROJECT_ROOT, 'data', 'images')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print('Project root:', PROJECT_ROOT)
print('CSV path   :', DATA_CSV, '— exists:', os.path.exists(DATA_CSV))

## 1. Load Dataset

In [ ]:
df = pd.read_csv(DATA_CSV)
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(5)

## 2. Schema Validation

The model requires exactly 8 columns. Verify all are present and typed correctly.

In [ ]:
REQUIRED_COLUMNS = {
    'text_content'   : 'Text input for the language encoder',
    'image_filename' : 'Image file referenced from data/images/',
    'user_age'       : 'Context feature — poster age (0 if unknown)',
    'past_flags'     : 'Context feature — prior flag count',
    'aggression'     : 'Label: aggressive language (0/1)',
    'repetition'     : 'Label: repeated harassment (0/1)',
    'intent'         : 'Label: clear harmful intent (0/1)',
    'language'       : 'Language tag (en / ur / roman_urdu)',
}

schema_rows = []
for col, desc in REQUIRED_COLUMNS.items():
    present = col in df.columns
    dtype   = str(df[col].dtype) if present else 'MISSING'
    nulls   = int(df[col].isnull().sum()) if present else -1
    schema_rows.append({'Column': col, 'Present': '✅' if present else '❌',
                        'Dtype': dtype, 'Null Count': nulls, 'Description': desc})

schema_df = pd.DataFrame(schema_rows).set_index('Column')
display(schema_df)

In [ ]:
# Detailed dtypes and memory
df.info()

## 3. Language Distribution

The model is designed for three languages: English (`en`), Urdu (`ur`), and Roman Urdu (`roman_urdu`).

In [ ]:
lang_counts = df['language'].value_counts()
lang_pct    = (lang_counts / len(df) * 100).round(1)

lang_summary = pd.DataFrame({'Count': lang_counts, 'Percentage (%)': lang_pct})
print('Language distribution:')
display(lang_summary)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(lang_counts.index, lang_counts.values,
              color=['#4C72B0', '#DD8452', '#55A868'], edgecolor='white', linewidth=1.2)
ax.bar_label(bars, labels=[f'{v:,}\n({p}%)' for v, p in zip(lang_counts.values, lang_pct.values)],
             padding=4, fontsize=10)
ax.set_xlabel('Language')
ax.set_ylabel('Sample Count')
ax.set_title('Samples per Language', fontweight='bold')
ax.set_ylim(0, lang_counts.max() * 1.18)
plt.tight_layout()
plt.savefig('language_distribution.png', bbox_inches='tight')
plt.show()

## 4. Label Distribution

Three binary multi-labels are predicted independently:
- **Aggression** — Does the post contain aggressive language?
- **Repetition** — Is this part of a repeated harassment pattern?
- **Intent** — Does the post show clear harmful intent?

Class imbalance directly impacts model performance and loss weighting strategy.

In [ ]:
label_cols = ['aggression', 'repetition', 'intent']

label_stats = []
for col in label_cols:
    vc = df[col].value_counts().sort_index()
    pos = int(vc.get(1, 0))
    neg = int(vc.get(0, 0))
    total = pos + neg
    ratio = f"{pos/total*100:.1f}% positive" if total > 0 else 'N/A'
    label_stats.append({'Label': col.capitalize(), 'Positive (1)': pos,
                        'Negative (0)': neg, 'Total': total,
                        'Positive Rate': ratio,
                        'Imbalance Note': 'Imbalanced' if pos/total > 0.6 or pos/total < 0.4 else 'Balanced'})

label_df = pd.DataFrame(label_stats).set_index('Label')
display(label_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = {'0 — Negative': '#d9534f', '1 — Positive': '#5cb85c'}

for ax, col in zip(axes, label_cols):
    vc = df[col].value_counts().sort_index()
    labels_pie = [f'{int(k)} — {"Negative" if k==0 else "Positive"}' for k in vc.index]
    clrs = ['#d9534f' if k == 0 else '#5cb85c' for k in vc.index]
    wedges, texts, autotexts = ax.pie(
        vc.values, labels=labels_pie, colors=clrs,
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
    )
    for at in autotexts:
        at.set_fontsize(10)
    ax.set_title(col.capitalize(), fontweight='bold', pad=12)

fig.suptitle('Label Class Distribution', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('label_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Co-occurrence: how often are multiple labels positive at once?
df['label_sum'] = df[label_cols].sum(axis=1)
cooc = df['label_sum'].value_counts().sort_index()
print('Label co-occurrence (how many labels are 1 for each sample):')
for k, v in cooc.items():
    print(f'  {int(k)} label(s) positive: {v:,} samples ({v/len(df)*100:.1f}%)')

## 5. Text Length Analysis

Text length (in characters) directly affects tokenization: the model truncates at `MAX_LEN = 128` tokens.
Understanding the distribution helps assess how much content is lost to truncation.

In [ ]:
df['char_len'] = df['text_content'].dropna().apply(len)
df['word_count'] = df['text_content'].dropna().apply(lambda x: len(str(x).split()))

print('Character length statistics:')
display(df['char_len'].describe().rename('Characters').to_frame())

print('\nWord count statistics:')
display(df['word_count'].describe().rename('Words').to_frame())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(df['char_len'].dropna(), bins=40, color='#4C72B0', edgecolor='white', linewidth=0.5)
ax1.axvline(df['char_len'].median(), color='#DD8452', linestyle='--', linewidth=1.8,
            label=f'Median: {df["char_len"].median():.0f}')
ax1.axvline(df['char_len'].mean(), color='#C44E52', linestyle='-', linewidth=1.8,
            label=f'Mean: {df["char_len"].mean():.0f}')
ax1.set_xlabel('Character Count')
ax1.set_ylabel('Frequency')
ax1.set_title('Text Length Distribution (Characters)', fontweight='bold')
ax1.legend()

ax2.hist(df['word_count'].dropna(), bins=30, color='#55A868', edgecolor='white', linewidth=0.5)
ax2.axvline(df['word_count'].median(), color='#DD8452', linestyle='--', linewidth=1.8,
            label=f'Median: {df["word_count"].median():.0f}')
ax2.axvline(df['word_count'].mean(), color='#C44E52', linestyle='-', linewidth=1.8,
            label=f'Mean: {df["word_count"].mean():.0f}')
ax2.set_xlabel('Word Count')
ax2.set_ylabel('Frequency')
ax2.set_title('Text Length Distribution (Words)', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig('text_length_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Text length by aggression label
fig, ax = plt.subplots(figsize=(8, 4))
for label_val, label_name, color in [(0, 'Non-aggressive', '#4C72B0'), (1, 'Aggressive', '#C44E52')]:
    subset = df[df['aggression'] == label_val]['char_len'].dropna()
    ax.hist(subset, bins=30, alpha=0.6, label=f'{label_name} (n={len(subset):,})',
            color=color, edgecolor='white', linewidth=0.4)
ax.set_xlabel('Character Count')
ax.set_ylabel('Frequency')
ax.set_title('Text Length by Aggression Label', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('text_length_by_aggression.png', bbox_inches='tight')
plt.show()

# Numeric summary
print(df.groupby('aggression')['char_len'].describe().rename(index={0: 'Non-aggressive', 1: 'Aggressive'}))

## 6. Missing Value Analysis

In [ ]:
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(3)

missing_df = pd.DataFrame({'Null Count': nulls, 'Null %': null_pct})
display(missing_df)

# Show the actual row(s) with null text
null_rows = df[df['text_content'].isnull()]
if len(null_rows) > 0:
    print(f'\n{len(null_rows)} row(s) with null text_content:')
    display(null_rows)
    print('\nAction: These rows will be skipped by the Dataset class (str() cast returns "nan") but should be dropped before training.')
else:
    print('No null text rows found.')

## 7. Context Features (user_age, past_flags)

In [ ]:
print('user_age distribution:')
display(df['user_age'].value_counts().head(10).to_frame())

print('\npast_flags distribution:')
display(df['past_flags'].value_counts().head(10).to_frame())

print('\nNote: Both context features appear to be constant (0.0) in this dataset.')
print('The context MLP branch will learn a fixed transformation and contribute minimally to predictions.')

## 8. Image Filename Analysis

In [ ]:
img_unique = df['image_filename'].unique()
print(f'Unique image filenames: {len(img_unique)}')
print(f'Values: {img_unique}')

# Check which images actually exist
found, missing = 0, 0
for fname in df['image_filename']:
    fpath = os.path.join(IMAGE_DIR, str(fname))
    if os.path.exists(fpath):
        found += 1
    else:
        missing += 1

print(f'\nImages resolved: {found:,} found / {missing:,} missing')
if missing == 0:
    print('✅ All image references resolve to files in data/images/')
else:
    print(f'⚠️  {missing} rows will fall back to blank 224×224 images at training time.')

## 9. Sample Records by Label

In [ ]:
print('=== Sample AGGRESSIVE posts (aggression=1) ===')
for _, row in df[df['aggression'] == 1][['text_content', 'aggression']].sample(5, random_state=42).iterrows():
    print(f'  [{row["aggression"]}] {row["text_content"]}')

print('\n=== Sample NON-AGGRESSIVE posts (aggression=0) ===')
for _, row in df[df['aggression'] == 0][['text_content', 'aggression']].sample(5, random_state=42).iterrows():
    print(f'  [{row["aggression"]}] {row["text_content"]}')

## 10. Key Findings & Observations

| Finding | Detail | Impact on Training |
|---|---|---|
| **Dataset size** | 2,000 samples | Small — cross-validation recommended (see PENDING_WORK §3.3) |
| **Language** | 100% Roman Urdu | Multilingual design is ready; future data can add English/Urdu |
| **Aggression label** | 72.1% positive, 27.9% negative | Class imbalance — consider `pos_weight` in `BCEWithLogitsLoss` |
| **Repetition label** | 100% negative (no positives) | Model cannot learn this label — output will always predict 0 |
| **Intent label** | 100% negative (no positives) | Same as Repetition — label is non-informative |
| **Image modality** | All rows use `none.jpg` placeholder | ResNet50 branch sees constant input; attention will down-weight it |
| **Context features** | All `user_age=0`, `past_flags=0` | Context MLP will learn a constant; contributes minimally |
| **Missing text** | 1 null value in `text_content` | Drop this row before training |
| **Text length** | Mean 68 chars; max 416 chars | Mostly within `MAX_LEN=128` token limit; minimal truncation expected |

In [ ]:
# Recommended: Drop the null row before training
df_clean = df.dropna(subset=['text_content']).reset_index(drop=True)
print(f'Rows before drop: {len(df):,}')
print(f'Rows after drop : {len(df_clean):,}')
print('\nRecommendation: Save cleaned CSV and update config.py to point to it.')
# df_clean.to_csv(os.path.join(PROJECT_ROOT, 'data', 'train_clean.csv'), index=False)